# Hidden Markov Models and Viterbi Decoding for POS Tagging

Today we will model language as a sequential probabilistic system.

Instead of treating words independently, we assume:
- Words are generated by hidden syntactic states (POS tags)
- Tags transistion sequentially according to learned probabilities

We will implement:
- A Hidden Markov Model (HMM)
- Parameter estimation from labeled data
- The Viterbi dynamic programming algorithm for decoding

### Hidden Markov Model Components

We define:
- Hidden states S = POS tags
- Observations O = words

We estimate:
- Transition probabilities: P(tag_t | tag_{t-1})
- Emission probabilities: P(word_t | tag_t)
- Initial probabilities: P(tag_1)

### Constraints

Use Laplace smoothing (+1) for stability.

### Expected Output
```
HMM POS TAGGER (VITERBI DECODER v1)

TRAINING STATISTICS
Number of Tags: ...
Vocabulary Size: ...

MODEL PARAMETERS
Transition Matrix Shape: (|S| × |S|)
Emission Coverage: ...

INPUT SENTENCE:
the dense matrix aligned meaning

PREDICTED TAG SEQUENCE:
[DET, ADJ, NOUN, VERB, NOUN]

TOKEN ↔ TAG ALIGNMENT:
the → DET
dense → ADJ
matrix → NOUN
aligned → VERB
meaning → NOUN
```

### Imports

In [51]:
from collections import defaultdict
import numpy as np

### Corpus (Training Data)

In [52]:
training_corpus = [
    [("the", "DET"), ("vector", "NOUN"), ("models", "VERB"), ("meaning", "NOUN")],
    [("a", "DET"), ("dense", "ADJ"), ("matrix", "NOUN"), ("captures", "VERB"), ("text", "NOUN")],
    [("the", "DET"), ("linear", "ADJ"), ("maps", "NOUN"), ("aligned", "VERB"), ("meaning", "NOUN")]
]

test_sentence = "the dense matrix aligned meaning"

### Vocabulary Construction

In [53]:
def build_vocabulary(corpus):
    """Builds a global vocabulary of all observed words in the training corpus."""

    vocab = set()

    for sentence in corpus:
        for word, _ in sentence:
            vocab.add(word)

    vocab.add("<UNK>")  # Explicit unknown token

    return vocab

### HMM Parameter Estimator (Training Engine)

In [54]:
def train_hmm(corpus):
    """Builds raw frequency counts for HMM training"""

    tag_counts = defaultdict(int)
    transition_counts = defaultdict(lambda: defaultdict(int))
    emission_counts = defaultdict(lambda: defaultdict(int))
    start_counts = defaultdict(int)

    for sentence in corpus:
        # Initial tab
        start_counts[sentence[0][1]] += 1

        for i, (word, tag) in enumerate(sentence):
            tag_counts[tag] += 1 # Count tag occurrences
            emission_counts[tag][word] += 1 # Count emissions
            
            # Count transitions
            if i > 0:
                prev_tag = sentence[i - 1][1]
                transition_counts[prev_tag][tag] += 1

    # Add UNK emission support per tag
    for tag in tag_counts:
        emission_counts[tag]["<UNK>"] += 1

    return tag_counts, transition_counts, emission_counts, start_counts

### Probability Computation Engine

In [55]:
def compute_probabilities(
        tag_counts, transition_counts, emission_counts, start_counts, vocab
    ):
    """Converts raw counts into probability distributions."""

    tags = list(tag_counts.keys())

    A = defaultdict(dict)  # Transition probabilities
    B = defaultdict(dict)  # Emission probabilities
    pi = {}                # Initial probabilities

    V = len(vocab)  # GLOBAL vocabulary size

    total_starts = sum(start_counts.values())

    # Initial probabilities π
    for tag in tags:
        pi[tag] = (start_counts[tag] + 1) / (total_starts + len(tags))

    # Transition probabilities A
    for prev_tag in tags:
        total_trans = sum(transition_counts[prev_tag].values())

        for next_tag in tags:
            A[prev_tag][next_tag] = (
                transition_counts[prev_tag][next_tag] + 1
            ) / (total_trans + len(tags))

    # Emission probabilities B
    for tag in tags:
        total_emit = sum(emission_counts[tag].values())

        for word in vocab:
            B[tag][word] = (
                emission_counts[tag].get(word, 0) + 1
            ) / (total_emit + len(vocab))

    return A, B, pi, tags

### Viterbi Decoding Engine

In [56]:
def viterbi(sentence, A, B, pi, tags):
    """
    Finds the most likely POS tag sequence for a sentence.
    """

    words = sentence.split()
    T = len(words)
    S = len(tags)

    V = np.zeros((S, T))
    backpointer = np.zeros((S, T), dtype=int)

    tag_to_idx = {tag: i for i, tag in enumerate(tags)}
    idx_to_tag = {i: tag for tag, i in tag_to_idx.items()}

    # Initialization
    for i, tag in enumerate(tags):
        emission_prob = B[tag].get(words[0], 1e-12)

        V[i, 0] = np.log(pi[tag]) + np.log(emission_prob)
        backpointer[i, 0] = 0

    # Recursion
    for t in range(1, T):
        for j, curr_tag in enumerate(tags):

            best_score = -np.inf
            best_state = 0

            emission_prob = B[curr_tag].get(words[t], 1e-12)

            for i, prev_tag in enumerate(tags):
                score = (
                    V[i, t - 1]
                    + np.log(A[prev_tag].get(curr_tag, 1e-6))
                    + np.log(emission_prob)
                )

                if score > best_score:
                    best_score = score
                    best_state = i

            V[j, t] = best_score
            backpointer[j, t] = best_state


    # Backtracking
    best_last = np.argmax(V[:, -1])
    best_path = [best_last]

    for t in range(T - 1, 0, -1):
        best_last = backpointer[best_last, t]
        best_path.append(best_last)

    best_path.reverse()

    return [idx_to_tag[i] for i in best_path]

### Evaluation Engine

In [57]:
def evaluate_model(sentence, tags_output):
    """Aligns words with predicted POS tags."""

    words = sentence.split()

    return list(zip(words, tags_output))

### Pipeline Orchestrator

In [58]:
def evaluate_hmm_pipeline(corpus, sentence):
    """
    Full HMM pipeline:
    Training -> Probability estimation -> Viterbi decoding -> Output formatting
    """
    
    print("HMM POS TAGGER (VITERBI DECODER v1)\n")

    # Build global vocabulary
    vocab = build_vocabulary(training_corpus)

    # Train HMM (count extraction)
    tag_counts, transition_counts, emission_counts, start_counts = train_hmm(training_corpus)

    # Convert counts → probability model (NOW includes vocab)
    A, B, pi, tags = compute_probabilities(
        tag_counts,
        transition_counts,
        emission_counts,
        start_counts,
        vocab
    )

    print("\nMODEL PARAMETERS")
    print(f"Transition Matrix Shape: ({len(tags)} × {len(tags)})")
    print(f"Emission Vocabulary Size: {len(vocab)}\n")

    # Run Viterbi decoding
    tags_output = viterbi(test_sentence, A, B, pi, tags)

    # Align tokens with predicted tags
    tagged_output = evaluate_model(test_sentence, tags_output)

    print("INPUT SENTENCE:")
    print(test_sentence)

    print("\nPREDICTED TAG SEQUENCE:")
    print(tags_output)

    print("\nTOKEN ↔ TAG ALIGNMENT:")
    for word, tag in tagged_output:
        print(f"{word} → {tag}")

### Execute Pipeline

In [59]:
evaluate_hmm_pipeline(training_corpus, test_sentence)

HMM POS TAGGER (VITERBI DECODER v1)


MODEL PARAMETERS
Transition Matrix Shape: (4 × 4)
Emission Vocabulary Size: 13

INPUT SENTENCE:
the dense matrix aligned meaning

PREDICTED TAG SEQUENCE:
['DET', 'ADJ', 'NOUN', 'VERB', 'NOUN']

TOKEN ↔ TAG ALIGNMENT:
the → DET
dense → ADJ
matrix → NOUN
aligned → VERB
meaning → NOUN
